# 🧪 Sesión 17 · Control de Acceso Basado en Roles (RBAC)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Qué diferencia hay entre no estar autenticado y no tener permiso?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/admin/panel   Authorization: Bearer <token_admin>
HTTP/1.1 200 OK
{"panel":"admin"}

GET /api/admin/panel   Authorization: Bearer <token_user>
HTTP/1.1 403 Forbidden

GET /api/admin/panel   (sin token)
HTTP/1.1 401 Unauthorized

GET /api/perfil        Authorization: Bearer <token_user>
HTTP/1.1 200 OK
{"usuario":"ana","rol":"user"}
```

_El 403 no es un error de credenciales: es una decisión de autorización._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Incluir el rol dentro de los claims del token.
- Distinguir 401 (no autenticado) de 403 (sin permiso).
- Proteger una ruta administrativa con RBAC.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El administrador obtiene su token → HTTP 200
- [ ] El usuario normal obtiene su token → HTTP 200
- [ ] El admin entra al panel → HTTP 200
- [ ] El usuario normal recibe 403 → HTTP 403
- [ ] Sin token recibe 401 → HTTP 401
- [ ] El perfil funciona para el usuario normal → HTTP 200

## 🧩 El reto

Implementa autenticación con roles: un usuario normal y un administrador. El panel administrativo solo debe abrirse para admin.

**Requisitos del contrato**

- Usuarios de prueba: admin/admin123 (rol admin) y ana/secreta (rol user).
- El token incluye el rol del usuario en sus claims.
- GET /api/admin/panel responde 200 solo para rol admin.
- Un usuario autenticado con rol user recibe 403.
- Sin token se responde 401, no 403.
- GET /api/perfil funciona para cualquier usuario autenticado.

> **Pista conceptual:** Primero autentica (¿quién eres?) y después autoriza (¿tu rol alcanza?). Si devuelves 403 a quien no tiene token, estás mezclando las dos preguntas.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion17.md` con la matriz rol → permiso.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [2]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


In [3]:
import os
import secrets
os.environ["JWT_SECRET_KEY"] = secrets.token_urlsafe(32)
print("Clave secreta configurada correctamente")


Clave secreta configurada correctamente


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tus respuestas de autorización se verán así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


In [7]:
%%writefile app.py
import os
from datetime import datetime, timedelta, timezone
from fastapi import FastAPI, Depends, HTTPException, status, Request
from fastapi.security import OAuth2PasswordBearer
from pydantic import BaseModel
from jose import jwt, JWTError

app = FastAPI(title="API con Control de Acceso por Roles", version="1.0.0")

SECRET_KEY = os.getenv("JWT_SECRET_KEY")
if not SECRET_KEY:
    raise RuntimeError("Falta configurar JWT_SECRET_KEY")

ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

USUARIOS = {
    "admin": {"password": "admin123", "rol": "admin"},
    "ana": {"password": "secreta", "rol": "user"}
}

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/api/token")

# Modelo para recibir los datos como JSON
class LoginData(BaseModel):
    usuario: str
    password: str

def crear_token(usuario: str, rol: str):
    ahora = datetime.now(timezone.utc)
    expiracion = ahora + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    datos_token = {
        "usuario": usuario,
        "rol": rol,
        "exp": expiracion
    }
    return jwt.encode(datos_token, SECRET_KEY, algorithm=ALGORITHM)

def obtener_usuario_actual(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        usuario = payload.get("usuario")
        rol = payload.get("rol")
        if not usuario or not rol:
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )
        return {"usuario": usuario, "rol": rol}
    except JWTError:
        raise HTTPException(
            status_code=401,
            detail="Token inválido o expirado",
            headers={"WWW-Authenticate": "Bearer"}
        )

# ✅ Ahora acepta JSON en lugar de formulario
@app.post("/api/token")
def login(datos: LoginData):
    if datos.usuario not in USUARIOS or USUARIOS[datos.usuario]["password"] != datos.password:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas",
            headers={"WWW-Authenticate": "Bearer"}
        )
    rol = USUARIOS[datos.usuario]["rol"]
    access_token = crear_token(datos.usuario, rol)
    return {"access_token": access_token, "token_type": "bearer"}

@app.get("/api/perfil")
def ver_perfil(usuario_actual = Depends(obtener_usuario_actual)):
    return {
        "usuario": usuario_actual["usuario"],
        "rol": usuario_actual["rol"]
    }

@app.get("/api/admin/panel")
def panel_admin(usuario_actual = Depends(obtener_usuario_actual)):
    if usuario_actual["rol"] != "admin":
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="No tienes permiso para acceder a este recurso"
        )
    return {"panel": "admin"}


Overwriting app.py


In [9]:
%%writefile app.py
import os
from datetime import datetime, timedelta, timezone
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer
from fastapi import Form
from jose import jwt, JWTError

# Inicializar la aplicación
app = FastAPI(title="API con Control de Acceso por Roles", version="1.0.0")

# Configuración de seguridad
SECRET_KEY = os.getenv("JWT_SECRET_KEY")
if not SECRET_KEY:
    raise RuntimeError("Falta configurar JWT_SECRET_KEY")

ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Base de datos de usuarios de prueba
USUARIOS = {
    "admin": {"password": "admin123", "rol": "admin"},
    "ana": {"password": "secreta", "rol": "user"}
}

# Herramienta para leer el token
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/api/token")

# ─── Función para crear el token ───
def crear_token(usuario: str, rol: str):
    ahora = datetime.now(timezone.utc)
    expiracion = ahora + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    datos_token = {
        "usuario": usuario,
        "rol": rol,
        "exp": expiracion
    }
    return jwt.encode(datos_token, SECRET_KEY, algorithm=ALGORITHM)

# ─── Dependencia: obtener el usuario actual desde el token ───
def obtener_usuario_actual(token: str = Depends(oauth2_scheme)):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        usuario = payload.get("usuario")
        rol = payload.get("rol")
        if not usuario or not rol:
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )
        return {"usuario": usuario, "rol": rol}
    except JWTError:
        raise HTTPException(
            status_code=401,
            detail="Token inválido o expirado",
            headers={"WWW-Authenticate": "Bearer"}
        )

# ─── Endpoint para obtener token ───
@app.post("/api/token")
def login(usuario: str = Form(...), password: str = Form(...)):
    if usuario not in USUARIOS or USUARIOS[usuario]["password"] != password:
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas",
            headers={"WWW-Authenticate": "Bearer"}
        )
    rol = USUARIOS[usuario]["rol"]
    access_token = crear_token(usuario, rol)
    return {"access_token": access_token, "token_type": "bearer"}

# ─── Ruta del perfil: cualquiera autenticado puede verlo ───
@app.get("/api/perfil")
def ver_perfil(usuario_actual = Depends(obtener_usuario_actual)):
    return {
        "usuario": usuario_actual["usuario"],
        "rol": usuario_actual["rol"]
    }

# ─── Ruta del panel de administración: SOLO para admin ───
@app.get("/api/admin/panel")
def panel_admin(usuario_actual = Depends(obtener_usuario_actual)):
    if usuario_actual["rol"] != "admin":
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="No tienes permiso para acceder a este recurso"
        )
    return {"panel": "admin"}


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Usamos FastAPI con Python en Google Colab. Necesitamos librerías para JWT, formularios HTTP y el motor de verificación del curso. Todo funciona en la nube, no instalo nada en mi PC.|
| **Rol** | Soy quien diseña y construye la parte de seguridad de una API: decido quién entra, qué puede ver cada persona y qué mensajes de error mostrar cuando alguien no puede pasar. |
| **Especificación** | Dos usuarios: admin/admin123 y ana/secreta. Cada uno recibe un token con su nombre y su rol. El panel de administración solo lo ve el admin; el usuario normal ve su perfil pero no el panel y recibe 403 si intenta entrar. Sin token se devuelve 401. |
| **Output** | Un archivo app.py que funciona sin errores: devuelve los códigos 200, 401 y 403 correctos, y el token incluye el rol de cada persona. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [8]:
from verificar import probar_http

casos = [{'descripcion': 'El administrador obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'admin', 'password': 'admin123'},
  'estado': 200,
  'guardar': {'token_admin': 'access_token'}},
 {'descripcion': 'El usuario normal obtiene su token',
  'metodo': 'POST',
  'ruta': '/api/token',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'guardar': {'token_user': 'access_token'}},
 {'descripcion': 'El admin entra al panel',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_admin}'},
  'estado': 200,
  'json_contiene': {'panel': 'admin'}},
 {'descripcion': 'El usuario normal recibe 403',
  'metodo': 'GET',
  'ruta': '/api/admin/panel',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 403},
 {'descripcion': 'Sin token recibe 401', 'metodo': 'GET', 'ruta': '/api/admin/panel', 'estado': 401},
 {'descripcion': 'El perfil funciona para el usuario normal',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token_user}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'rol': 'user'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 6 casos · app.py
✅ 01. El administrador obtiene su token
✅ 02. El usuario normal obtiene su token
✅ 03. El admin entra al panel
✅ 04. El usuario normal recibe 403
✅ 05. Sin token recibe 401
✅ 06. El perfil funciona para el usuario normal
------------------------------------------------------------------------------
RESULTADO: 6/6 casos correctos. ¡Resultado logrado!


In [11]:
from verificar import probar_http

casos = [
    {
        "descripcion": "El administrador obtiene su token",
        "metodo": "POST",
        "ruta": "/api/token",
        "data": {"usuario": "admin", "password": "admin123"},
        "estado": 200,
        "guardar": {"token_admin": "access_token"}
    },
    {
        "descripcion": "El usuario normal obtiene su token",
        "metodo": "POST",
        "ruta": "/api/token",
        "data": {"usuario": "ana", "password": "secreta"},
        "estado": 200,
        "guardar": {"token_user": "access_token"}
    },
    {
        "descripcion": "El admin entra al panel",
        "metodo": "GET",
        "ruta": "/api/admin/panel",
        "headers": {"Authorization": "Bearer {token_admin}"},
        "estado": 200,
        "json_contiene": {"panel": "admin"}
    },
    {
        "descripcion": "El usuario normal recibe 403",
        "metodo": "GET",
        "ruta": "/api/admin/panel",
        "headers": {"Authorization": "Bearer {token_user}"},
        "estado": 403
    },
    {
        "descripcion": "Sin token recibe 401",
        "metodo": "GET",
        "ruta": "/api/admin/panel",
        "estado": 401
    },
    {
        "descripcion": "El perfil funciona para el usuario normal",
        "metodo": "GET",
        "ruta": "/api/perfil",
        "headers": {"Authorization": "Bearer {token_user}"},
        "estado": 200,
        "json_contiene": {"usuario": "ana", "rol": "user"}
    }
]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 6 casos · app.py
✅ 01. El administrador obtiene su token
✅ 02. El usuario normal obtiene su token
✅ 03. El admin entra al panel
✅ 04. El usuario normal recibe 403
✅ 05. Sin token recibe 401
✅ 06. El perfil funciona para el usuario normal
------------------------------------------------------------------------------
RESULTADO: 6/6 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion17.md` a `auditorias/auditoria_sesion17.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué? R.- Lo que más me costó fue entender bien la diferencia entre devolver un 401 y un 403. Al principio me confundía porque ambos significan "no puedes pasar", pero en distintos momentos. Tuve que repetir varias veces en mi mente: primero verifico si hay token (401 si no hay), y solo después de saber quién es, reviso si tiene permiso (403 si no lo tiene). Si lo hago al revés, le digo a alguien que no existe que "no tiene permiso", y eso no es correcto.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato? R.- Decidí guardar los usuarios de prueba en un diccionario sencillo al inicio del archivo, en lugar de usar una base de datos. También creé una función `obtener_usuario_actual` que uso como "cerrojo" en mis rutas protegidas; así no repito el mismo código de validación del token en cada endpoint. No me lo pedían, pero hace que el código sea más limpio y fácil de leer.
   - ¿Qué error detectó la verificación que tú no habías previsto? R.- En mi primer intento, cuando alguien intentaba entrar al panel de administración sin token, yo devolvía 403 en lugar de 401. La verificación me marcó ese error. No me había dado cuenta de que el orden importa: primero debes saber quién es la persona antes de juzgar si tiene permiso. El verificador también comprobó que el rol viniera dentro del token y que el perfil devolviera tanto el nombre como el rol; yo solo regresaba el nombre y me faltaba incluir el rol en la respuesta.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
